# 02 — ModernBERT a parità di genere (Versione 3)

La ricetta della Versione 2 (2 epoche fisse, 1024 token, 20 articoli per testata a ogni epoca, al più 5 testate per
gruppo a ogni epoca con il tetto a rotazione, batch piccolo con accumulo fino a 32), con due differenze:

- compito **binario** (affidabile = punteggio ≥ 60): due uscite;
- **pesi a parità di genere** nella loss: ogni articolo pesa il fattore della sua coppia (orientamento, classe), in
  modo che dentro ogni orientamento affidabili e inaffidabili contino uguale.

Valutazione come `01_feature_genere`: testate valutate, media delle probabilità, metrica principale **media sugli
orientamenti** dell'accuratezza bilanciata. A ogni fold finito il log scrive anche il TF-IDF sullo **stesso fold**
(se `feature_genere` è già finito): si vede a metà dell'opera come va.

Se la memoria GPU non basta (server condiviso, NVML rotto) il fold riparte con il batch dimezzato.

In [ ]:
import gc
import json
import math
import random
import time

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import StratifiedGroupKFold
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

from comune import RESULTS, carica, fattori_genere, metriche, per_testata, registratore

In [ ]:
NOME = "modernbert_base_genere_s42"
MODELLO = "answerdotai/ModernBERT-base"
MAX_TOKEN = 1024
BATCH = 4
BATCH_EFFETTIVO = 32
BATCH_EVAL = 16
ARTICOLI_PER_TESTATA = 20
MAX_TESTATE_PER_GRUPPO = 5
EPOCHE = 2
LR = 2e-5
WARMUP = 0.1
WEIGHT_DECAY = 0.01
SEED = 42
FP16 = True
PROVA = False

In [ ]:
def fissa_seed(s):
    s = int(s); random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


base = RESULTS / "prova" if PROVA else RESULTS
OUT = base / "transformer" / NOME
OUT.mkdir(parents=True, exist_ok=True)
registra = registratore(base / f"avanzamento_{NOME}.log")
if PROVA:
    EPOCHE = 1
json.dump({k: globals()[k] for k in ["NOME", "MODELLO", "MAX_TOKEN", "BATCH", "BATCH_EFFETTIVO", "ARTICOLI_PER_TESTATA",
                                     "MAX_TESTATE_PER_GRUPPO", "EPOCHE", "LR", "WARMUP", "WEIGHT_DECAY", "SEED", "PROVA"]},
          open(OUT / "config.json", "w"), indent=1)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
registra(f"=== {NOME}: {MODELLO}, seed {SEED}, su {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'} ===")
art = carica(PROVA)
tok = AutoTokenizer.from_pretrained(MODELLO)
ids = tok(art.text.tolist(), truncation=True, max_length=MAX_TOKEN)["input_ids"]
lunghezze = np.array([len(x) for x in ids]); PAD = tok.pad_token_id
registra(f"{len(art)} articoli tokenizzati, {art.domain.nunique()} testate")
RIF = base / "feature_genere" / "articoli.csv"


def lotto(indici):
    L = max(lunghezze[i] for i in indici)
    x = np.full((len(indici), L), PAD, dtype=np.int64); m = np.zeros((len(indici), L), dtype=np.int64)
    for r, i in enumerate(indici):
        x[r, :lunghezze[i]] = ids[i]; m[r, :lunghezze[i]] = 1
    return torch.from_numpy(x).to(device), torch.from_numpy(m).to(device)


@torch.no_grad()
def predici(model, indici):
    model.eval(); indici = np.asarray(indici); ordine = np.argsort(lunghezze[indici])
    out = np.zeros((len(indici), 2), dtype=np.float32)
    for s in range(0, len(indici), BATCH_EVAL):
        pezzo = ordine[s:s + BATCH_EVAL]; x, m = lotto(indici[pezzo])
        with torch.autocast("cuda", dtype=torch.float16, enabled=FP16):
            out[pezzo] = model(input_ids=x, attention_mask=m).logits.float().cpu().numpy()
    return out


def tabella(indici, logit):
    s = art.loc[indici]
    p = torch.softmax(torch.from_numpy(logit), 1).numpy()[:, 1]
    return pd.DataFrame({"article_id": s.article_id.values, "domain": s.domain.values, "fold": s.fold_gruppo.values, "y": s.y.values,
                         "orientamento": s.orientamento.values, "logit": logit[:, 1] - logit[:, 0], "p": p})

In [ ]:
def un_fold(fold):
    fissa_seed(SEED + fold); t_fold = time.time()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    te = art.index[(art.fold_gruppo == fold) & art.valutazione]
    tr_tutti = art[art.fold_gruppo != fold]
    t_tr = tr_tutti.drop_duplicates("domain").reset_index(drop=True)
    _, iv = next(StratifiedGroupKFold(7, shuffle=True, random_state=SEED).split(t_tr, t_tr.orientamento + t_tr.y.astype(str), t_tr.gruppo))
    in_val = set(t_tr.domain.iloc[iv])
    tr = tr_tutti[~tr_tutti.domain.isin(in_val)]
    va = tr_tutti.index[tr_tutti.domain.isin(in_val) & tr_tutti.valutazione]
    articoli_di = {d: g.index.to_numpy() for d, g in tr.groupby("domain")}
    testate_di = {g: sorted(x.domain.unique()) for g, x in tr.groupby("gruppo")}
    fattori = fattori_genere(tr.drop_duplicates("domain"))           # testate pesate (un gruppo vale al più 5)
    w_art = np.array([fattori.get((o, c), 1.0) for o, c in zip(art.orientamento, art.y)], dtype=np.float32)

    def campione(epoca):
        rng = np.random.default_rng(SEED * 1000 + fold * 100 + epoca); pezzi = []
        for g in sorted(testate_di):
            ds = testate_di[g]
            for d in (rng.choice(ds, MAX_TESTATE_PER_GRUPPO, replace=False) if len(ds) > MAX_TESTATE_PER_GRUPPO else ds):
                pezzi.append(rng.choice(articoli_di[d], min(len(articoli_di[d]), ARTICOLI_PER_TESTATA), replace=False))
        c = np.concatenate(pezzi); rng.shuffle(c); return c

    campioni = {e: campione(e) for e in range(1, EPOCHE + 1)}
    registra(f"fold {fold}: {len(articoli_di)} testate di training, {len(in_val)} di validazione, {art.loc[te].domain.nunique()} di test; "
             f"{len(campioni[1])} articoli per epoca; batch {BATCH}")
    model = AutoModelForSequenceClassification.from_pretrained(MODELLO, num_labels=2, reference_compile=False).to(device)
    nod = ["bias", "LayerNorm.weight", "norm.weight"]
    opt = torch.optim.AdamW([{"params": [p for n, p in model.named_parameters() if not any(x in n for x in nod)], "weight_decay": WEIGHT_DECAY},
                             {"params": [p for n, p in model.named_parameters() if any(x in n for x in nod)], "weight_decay": 0.0}], lr=LR)
    acc = max(1, BATCH_EFFETTIVO // BATCH)
    passi = sum(math.ceil(len(c) / (BATCH * acc)) for c in campioni.values())
    sched = get_linear_schedule_with_warmup(opt, int(WARMUP * passi), passi)
    scaler = torch.amp.GradScaler("cuda", enabled=FP16)
    righe = []
    for epoca in range(1, EPOCHE + 1):
        t0 = time.time(); c = campioni[epoca]; model.train(); tot, n = 0.0, 0; opt.zero_grad()
        for s in range(0, len(c), BATCH):
            pezzo = c[s:s + BATCH]; x, m = lotto(pezzo)
            yb = torch.tensor(art.y.values[pezzo], device=device); wb = torch.tensor(w_art[pezzo], device=device)
            with torch.autocast("cuda", dtype=torch.float16, enabled=FP16):
                logit = model(input_ids=x, attention_mask=m).logits.float()
            loss = (torch.nn.functional.cross_entropy(logit, yb, reduction="none") * wb).sum() / wb.sum()
            scaler.scale(loss / acc).backward(); tot += loss.item() * len(pezzo); n += len(pezzo)
            if (s // BATCH + 1) % acc == 0 or s + BATCH >= len(c):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
        mv = metriche(per_testata(tabella(va, predici(model, va)))) if len(va) else {"media sugli orientamenti": float("nan")}
        righe.append({"fold": fold, "epoca": epoca, "loss_train": tot / n, "val_media_orientamenti": mv["media sugli orientamenti"], "secondi": time.time() - t0})
        registra(f"fold {fold} epoca {epoca}/{EPOCHE}: loss {tot / n:.4f}, validazione: media sugli orientamenti {mv['media sugli orientamenti']:.3f} ({time.time() - t0:.0f} s)")
    t_test = tabella(te, predici(model, te))
    pd.DataFrame(righe).to_csv(OUT / f"fold{fold}_epoche.csv", index=False)
    t_test.to_csv(OUT / f"fold{fold}_test.csv", index=False)
    m = metriche(per_testata(t_test))
    confronto = ""
    if RIF.exists():
        r = pd.read_csv(RIF); r = r[r.fold == fold]
        mr = metriche(per_testata(r))
        confronto = f" | TF-IDF sullo stesso fold: {mr['media sugli orientamenti']:.3f} (AUC {mr['AUC media sugli orientamenti']:.3f})"
    registra(f"fold {fold}: finito in {(time.time() - t_fold) / 60:.0f} min; media sugli orientamenti {m['media sugli orientamenti']:.3f} "
             f"(AUC {m['AUC media sugli orientamenti']:.3f}){confronto}; memoria GPU massima "
             f"{torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0:.1f} GB")
    del model, opt, sched, scaler; gc.collect(); torch.cuda.empty_cache()

In [ ]:
folds = sorted(int(f) for f in art.fold_gruppo.unique())
for fold in folds:
    if (OUT / f"fold{fold}_test.csv").exists():
        registra(f"fold {fold}: già fatto, salto"); continue
    while True:
        memoria_finita = False
        try:
            un_fold(fold)
        except RuntimeError as e:                 # OutOfMemoryError o, con NVML rotto, "INTERNAL ASSERT ... CUDACachingAllocator"
            if not any(x in str(e) for x in ["out of memory", "CUBLAS_STATUS_ALLOC_FAILED", "CUDA error", "CUDACachingAllocator", "NVML"]):
                raise
            registra(f"fold {fold}: memoria GPU finita ({str(e)[:120]})"); memoria_finita = True
        if not memoria_finita:
            break
        gc.collect(); torch.cuda.empty_cache()
        if BATCH == 1:
            raise RuntimeError("memoria GPU insufficiente anche con batch 1")
        BATCH, BATCH_EVAL = max(1, BATCH // 2), max(1, BATCH_EVAL // 2)
        registra(f"fold {fold}: riparto con batch {BATCH}")
pred = pd.concat([pd.read_csv(OUT / f"fold{f}_test.csv") for f in folds if (OUT / f"fold{f}_test.csv").exists()])
if len(pred) == int(art.valutazione.sum()):
    pred.to_csv(OUT / "articoli.csv", index=False)
    m = metriche(per_testata(pred)); json.dump(m, open(OUT / "metriche.json", "w"), indent=1)
    (OUT / "FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))
    registra(f"=== {NOME} FINITO: media sugli orientamenti {m['media sugli orientamenti']:.3f}, AUC {m['AUC media sugli orientamenti']:.3f}, "
             f"bilanciata su tutto {m['bilanciata su tutto']:.3f} ===")